# M2-E3 Qwen Parser — LoRA Fine-Tuning (Colab)

Fine-tunes **Qwen2.5-0.5B-Instruct** with LoRA (PEFT) on pre-built training splits to produce a robust 8-intent frame/intent parser for M2.

> **Runtime**: `Runtime → Change runtime type → T4 GPU`

## Prerequisites (run locally BEFORE opening this notebook)

```bash
# 1. Merge gold + additional batches, validate, build prompts, split 80/10/10
python scripts/m2_e3/prepare_qwen_data.py

# 2. Upload the output folder to your Drive:
#    experiments/m2_e3_parse/data/splits_qwen/
#    experiments/m2_e3_parse/data/temporal_queries_merged.jsonl
```

## Notebook Steps
1. Mount Drive & configure paths
2. Install & verify dependencies
3. Verify pre-built splits
4. Load Qwen2.5-0.5B-Instruct + LoRA adapter
5. Tokenise dataset
6. Train (~25 min on T4)
7. Save adapter to Drive
8. Evaluate — overall + per-intent breakdown
9. Export zip for local download

## 0 · Colab Detection & Drive Mount

In [1]:
import sys

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('Not in Colab — skipping mount.')

Mounted at /content/drive
Google Drive mounted.


## 1 · Configuration

**Only change `DRIVE_REPO_PATH`** — point it to the folder in your Drive where you uploaded the repo (the one containing `experiments/`).

In [2]:
import os, pathlib

# ── EDIT THIS ────────────────────────────────────────────────────────
DRIVE_REPO_PATH = '/content/drive/MyDrive/snet_related/Natural-Language-Explainability-for-Temporal-KGs'
# ─────────────────────────────────────────────────────────────────────

BASE_DIR    = pathlib.Path(DRIVE_REPO_PATH) if IN_COLAB else pathlib.Path('.').resolve().parent
DATA_DIR    = BASE_DIR / 'experiments' / 'm2_e3_parse' / 'data'
SPLITS_DIR  = DATA_DIR / 'splits_qwen'
ADAPTER_DIR = BASE_DIR / 'experiments' / 'm2_e3_parse' / 'artifacts' / 'qwen_parser_lora'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME    = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SEQ_LEN   = 256
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.10
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3  # val loss bottoms at epoch 2; early-stop guards the rest
BATCH_SIZE    = 8
GRAD_ACCUM    = 4
SEED          = 42

print('BASE_DIR    :', BASE_DIR)
print('SPLITS_DIR  :', SPLITS_DIR)
print('Splits exist:', SPLITS_DIR.exists())

BASE_DIR    : /content/drive/MyDrive/snet_related/Natural-Language-Explainability-for-Temporal-KGs
SPLITS_DIR  : /content/drive/MyDrive/snet_related/Natural-Language-Explainability-for-Temporal-KGs/experiments/m2_e3_parse/data/splits_qwen
Splits exist: True


## 2 · Install & Verify Dependencies

Pinned versions avoid ABI mismatches between bitsandbytes / PEFT / accelerate.

In [3]:
!pip install transformers peft accelerate bitsandbytes>=0.46.1 datasets==3.0.2 trl==0.12.1 scikit-learn evaluate==0.4.3 tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [4]:
import torch, transformers, peft
print('torch       :', torch.__version__)
print('CUDA        :', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('peft        :', peft.__version__)
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))
    print('VRAM (GB)   :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

torch       : 2.11.0+cu128
CUDA        : True
transformers: 5.12.0
peft        : 0.19.1
GPU         : Tesla T4
VRAM (GB)   : 15.6


## 3 · Verify Pre-Built Splits

These files were created locally by `scripts/m2_e3/prepare_qwen_data.py`.
If the check below fails, re-run the local script and re-upload `splits_qwen/`.

In [5]:
import json
from pathlib import Path
from collections import Counter

def read_jsonl(p):
    rows = []
    with open(p, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: rows.append(json.loads(line))
                except: pass
    return rows

split_counts = {}
for split in ['train', 'val', 'test']:
    p = SPLITS_DIR / f'{split}_prompts.jsonl'
    if not p.exists():
        raise FileNotFoundError(f'Missing: {p}  — run prepare_qwen_data.py locally first!')
    rows = read_jsonl(p)
    split_counts[split] = len(rows)
    c = Counter(json.loads(r['target']).get('intent', '?') for r in rows)
    print(f'{split} ({len(rows):,}): ' + '  '.join(f'{k}={v}' for k,v in sorted(c.items())))

# Load stats.json if present
stats_path = SPLITS_DIR / 'stats.json'
if stats_path.exists():
    stats = json.loads(stats_path.read_text())
    print('\nMerged total:', stats.get('total_merged', '?'))
    print('Sources:', stats.get('sources', {}))

print('\n✓ Splits verified — ready to train.')

train (1,680): AGG=344  CAUSAL=195  COMPARE=94  INTERVAL=144  OVERLAP=86  POINT=275  PREDICT=218  SEQUENCE=324
val (210): AGG=53  CAUSAL=22  COMPARE=18  INTERVAL=10  OVERLAP=12  POINT=29  PREDICT=24  SEQUENCE=42
test (210): AGG=44  CAUSAL=26  COMPARE=14  INTERVAL=18  OVERLAP=13  POINT=37  PREDICT=28  SEQUENCE=30

Merged total: 2100
Sources: {'base_gold': 1137, 'extra_batches': 963, 'duplicates_removed': 0}

✓ Splits verified — ready to train.


## 4 · Load Qwen2.5-0.5B-Instruct + LoRA

Uses **4-bit NF4 quantisation** (saves ~2 GB VRAM on T4). LoRA rank-16 targets all attention + MLP projection layers (~1.2% trainable params).

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

bnb_cfg = None
if DEVICE == 'cuda':
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_cfg,
    device_map='auto' if DEVICE == 'cuda' else None,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32,
)

if DEVICE == 'cuda':
    base_model = prepare_model_for_kbit_training(base_model)

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
if DEVICE == 'cuda':
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Device: cuda
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
VRAM used: 0.77 GB


## 5 · Tokenise Dataset

Prompt tokens are masked with `-100` so the loss only trains on the JSON target.

In [7]:
from torch.utils.data import Dataset

class ParserDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=MAX_SEQ_LEN):
        self.examples = []
        for row in read_jsonl(path):
            msgs = row['messages']
            prompt_text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
            target_text = row['target'] + tokenizer.eos_token
            full_text   = prompt_text + target_text
            enc = tokenizer(full_text, max_length=max_len, truncation=True,
                            padding='max_length', return_tensors='pt')
            input_ids = enc['input_ids'][0]
            attn_mask = enc['attention_mask'][0]
            prompt_len = tokenizer(prompt_text, return_tensors='pt')['input_ids'].shape[1]
            labels = input_ids.clone()
            labels[:prompt_len] = -100  # mask prompt — train on target only
            self.examples.append({
                'input_ids': input_ids,
                'attention_mask': attn_mask,
                'labels': labels,
            })

    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

print('Tokenising train split...')
train_dataset = ParserDataset(SPLITS_DIR / 'train_prompts.jsonl', tokenizer)
print('Tokenising val split...')
val_dataset   = ParserDataset(SPLITS_DIR / 'val_prompts.jsonl',   tokenizer)
print(f'Train: {len(train_dataset):,}  Val: {len(val_dataset):,}')

Tokenising train split...
Tokenising val split...
Train: 1,680  Val: 210


## 6 · Train with SFTTrainer

| Setting | Value |
|---|---|
| Epochs | 4 |
| Effective batch | 32 (8 × 4 grad accum) |
| LR | 2e-4 (cosine, 5% warmup) |
| Expected time | ~25 min on T4 |

> The best checkpoint (lowest eval loss) is loaded automatically at the end.

In [8]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, model=model,
        pad_to_multiple_of=8, label_pad_token_id=-100),
)

print('Starting training...')
trainer.train()
print('Training complete!')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.016746,0.009630
2,0.007561,0.008790
3,0.005603,0.008597


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete!


## 7 · Save LoRA Adapter to Drive

In [9]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print('Adapter saved to:', ADAPTER_DIR)
print('Files:', sorted(f.name for f in ADAPTER_DIR.iterdir()))

Adapter saved to: /content/drive/MyDrive/snet_related/Natural-Language-Explainability-for-Temporal-KGs/experiments/m2_e3_parse/artifacts/qwen_parser_lora
Files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'checkpoints', 'eval_results.jsonl', 'tokenizer.json', 'tokenizer_config.json']


## 8 · Evaluate on Test Split

In [10]:
from tqdm.auto import tqdm

model.eval()

def normalize_val(val):
    import re
    s = str(val).lower().strip()
    s = re.sub(r"\b(the|a|an)\b", "", s)
    s = re.sub(r'[.,\/#!$%\^&\*;:{}=\-_`~()?]', '', s)
    return re.sub(r"\s+", " ", s).strip()

def is_normalized_exact(p_str, g_str):
    try:
        p_obj = json.loads(p_str)
        g_obj = json.loads(g_str)
    except:
        return False
    if p_obj.get('intent') != g_obj.get('intent'):
        return False
    p_frame = p_obj.get('frame', {})
    g_frame = g_obj.get('frame', {})
    if set(p_frame.keys()) != set(g_frame.keys()):
        return False
    for k in g_frame:
        if normalize_val(g_frame[k]) != normalize_val(p_frame.get(k)):
            return False
    return True

def predict(row):
    msgs = row['messages']
    prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=128, do_sample=False,
            pad_token_id=tokenizer.eos_token_id)
    new_ids = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

test_examples = read_jsonl(SPLITS_DIR / 'test_prompts.jsonl')
n_exact = n_norm_exact = n_intent = 0
results = []

for ex in tqdm(test_examples, desc='Eval'):
    pred_str = predict(ex)
    gold_str = ex['target']
    exact = pred_str.strip() == gold_str.strip()
    n_exact += int(exact)
    norm_exact = is_normalized_exact(pred_str, gold_str)
    n_norm_exact += int(norm_exact)
    try:
        intent_ok = json.loads(pred_str).get('intent') == json.loads(gold_str).get('intent')
    except:
        intent_ok = False
    n_intent += int(intent_ok)
    results.append({'id': ex['id'], 'exact': exact, 'norm_exact': norm_exact, 'intent': intent_ok,
                    'pred': pred_str, 'gold': gold_str})

n_eval = len(results)
print(f'\nEval on {n_eval} examples:')
print(f'  Exact match (Strict)    : {n_exact}/{n_eval} = {100*n_exact/n_eval:.1f}%')
print(f'  Exact match (Normalized): {n_norm_exact}/{n_eval} = {100*n_norm_exact/n_eval:.1f}%')
print(f'  Intent match            : {n_intent}/{n_eval} = {100*n_intent/n_eval:.1f}%')


Eval:   0%|          | 0/210 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Eval on 210 examples:
  Exact match (Strict)    : 171/210 = 81.4%
  Exact match (Normalized): 177/210 = 84.3%
  Intent match            : 210/210 = 100.0%


In [11]:
# Per-intent accuracy breakdown
from collections import defaultdict

intent_stats = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in results:
    try: gold_intent = json.loads(r['gold']).get('intent', 'UNKNOWN')
    except: gold_intent = 'UNKNOWN'
    intent_stats[gold_intent]['total'] += 1
    if r['intent']: intent_stats[gold_intent]['correct'] += 1

print(f'{"Intent":<10}  {"Correct":>8}  {"Total":>8}  {"Acc%":>8}')
print('-' * 44)
for intent, s in sorted(intent_stats.items()):
    acc = 100 * s['correct'] / max(s['total'], 1)
    mark = '  OK' if acc >= 70 else '  LOW'
    print(f'{intent:<10}  {s["correct"]:>8}  {s["total"]:>8}  {acc:>7.1f}%{mark}')

Intent       Correct     Total      Acc%
--------------------------------------------
AGG               44        44    100.0%  OK
CAUSAL            26        26    100.0%  OK
COMPARE           14        14    100.0%  OK
INTERVAL          18        18    100.0%  OK
OVERLAP           13        13    100.0%  OK
POINT             37        37    100.0%  OK
PREDICT           28        28    100.0%  OK
SEQUENCE          30        30    100.0%  OK


## 8b - Frame-Field Accuracy Breakdown

Which specific frame fields drive the exact-match gap? Correct intent but mismatched field value = paraphrase, not a model failure.


In [12]:
# Frame-field accuracy breakdown — explains the exact-match gap
from collections import defaultdict
import re

def extract_year(val):
    if not val:
        return None
    m = re.search(r"\b\d{4}\b", str(val))
    return m.group(0) if m else None

def is_extractive(gold_val, question):
    if not gold_val or not question:
        return False
    g_norm = normalize_val(gold_val)
    q_norm = normalize_val(question)
    if not g_norm:
        return False
    year = extract_year(gold_val)
    if year and year in q_norm:
        return True
    return g_norm in q_norm

field_stats = defaultdict(lambda: {'correct': 0, 'total': 0})
time_stats = {
    'extractive_total': 0, 'extractive_correct': 0,
    'parametric_total': 0, 'parametric_correct': 0,
    'year_match_total': 0, 'year_match_correct': 0
}
parse_errors = 0

for r in results:
    try:
        gold_obj = json.loads(r['gold'])
        pred_obj = json.loads(r['pred'])
    except Exception:
        parse_errors += 1
        continue
    gold_frame = gold_obj.get('frame', {})
    pred_frame = pred_obj.get('frame', {})

    # Retrieve original question from test example
    qid = r['id']
    question = ""
    for ex in test_examples:
        if ex['id'] == qid:
            question = next((m['content'] for m in ex['messages'] if m['role'] == 'user'), "")
            break

    for field, gold_val in gold_frame.items():
        field_stats[field]['total'] += 1
        pred_val = pred_frame.get(field)

        is_correct = normalize_val(pred_val) == normalize_val(gold_val)
        if is_correct:
            field_stats[field]['correct'] += 1

        if field in ['time', 'date', 'period']:
            g_year = extract_year(gold_val)
            p_year = extract_year(pred_val)
            if g_year:
                time_stats['year_match_total'] += 1
                if g_year == p_year:
                    time_stats['year_match_correct'] += 1

            if is_extractive(gold_val, question):
                time_stats['extractive_total'] += 1
                if is_correct:
                    time_stats['extractive_correct'] += 1
            else:
                time_stats['parametric_total'] += 1
                if is_correct:
                    time_stats['parametric_correct'] += 1

header = f"{'Field':<20}  {'Correct':>8}  {'Total':>8}  {'Acc%':>8}  {'Miss%':>8}"
print(header)
print('-' * 60)
for field, s in sorted(field_stats.items(), key=lambda x: -x[1]['total']):
    acc  = 100 * s['correct'] / max(s['total'], 1)
    miss = 100 - acc
    mark = '  <-- paraphrase hotspot' if acc < 90 else ''
    row  = f"{field:<20}  {s['correct']:>8}  {s['total']:>8}  {acc:>7.1f}%  {miss:>6.1f}%{mark}"
    print(row)

print('\n' + '=' * 60)
print('Temporal / Date Extraction Analysis')
print('=' * 60)
if time_stats['extractive_total']:
    ext_acc = 100 * time_stats['extractive_correct'] / time_stats['extractive_total']
    print(f"Extractive time slots  : {time_stats['extractive_correct']}/{time_stats['extractive_total']} = {ext_acc:.1f}%")
if time_stats['parametric_total']:
    param_acc = 100 * time_stats['parametric_correct'] / time_stats['parametric_total']
    print(f"Parametric (Factoid)   : {time_stats['parametric_correct']}/{time_stats['parametric_total']} = {param_acc:.1f}%")
if time_stats['year_match_total']:
    year_acc = 100 * time_stats['year_match_correct'] / time_stats['year_match_total']
    print(f"Year-only matching     : {time_stats['year_match_correct']}/{time_stats['year_match_total']} = {year_acc:.1f}%")
print('=' * 60)

if parse_errors:
    print(f'\nJSON parse errors: {parse_errors}')

non_exact_intent_ok = [r for r in results if not r['exact'] and r['intent']]
print(f'\n{n_eval - n_exact} non-exact (Strict): {len(non_exact_intent_ok)} had correct intent '
      f'(frame-field paraphrase only) vs {n_eval - n_exact - len(non_exact_intent_ok)} '
      f'had wrong intent too.')

non_exact_norm_intent_ok = [r for r in results if not r['norm_exact'] and r['intent']]
print(f'{n_eval - n_norm_exact} non-exact (Normalized): {len(non_exact_norm_intent_ok)} had correct intent '
      f'vs {n_eval - n_norm_exact - len(non_exact_norm_intent_ok)} '
      f'had wrong intent too.')


Field                  Correct     Total      Acc%     Miss%
------------------------------------------------------------
metric                     125       131     95.4%     4.6%
period                      69        70     98.6%     1.4%
event                       36        43     83.7%    16.3%  <-- paraphrase hotspot
time                        15        37     40.5%    59.5%  <-- paraphrase hotspot
relation                    30        30    100.0%     0.0%
cause                       26        26    100.0%     0.0%
effect                      25        26     96.2%     3.8%
date                        21        21    100.0%     0.0%
anchor_event                20        20    100.0%     0.0%
region                      18        18    100.0%     0.0%
a                           14        14    100.0%     0.0%
b                           14        14    100.0%     0.0%
start                       12        12    100.0%     0.0%
end                         12        12    100.0%

In [13]:
# Show first 5 predictions for quick sanity check
for r in results[:5]:
    print('\n---', r['id'])
    print('GOLD:', r['gold'])
    print('PRED:', r['pred'])
    print('Match → exact:', r['exact'], ' intent:', r['intent'])


--- q1044
GOLD: {"intent": "POINT", "frame": {"event": "Declaration of Independence signing", "time": "1776-07-04"}}
PRED: {"intent": "POINT", "frame": {"event": "Declaration of Independence signing", "time": "1776-09-08"}}
Match → exact: False  intent: True

--- q1127
GOLD: {"intent": "SEQUENCE", "frame": {"metric": "latency", "anchor_event": "the patch", "relation": "after"}}
PRED: {"intent": "SEQUENCE", "frame": {"metric": "latency", "anchor_event": "the patch", "relation": "after"}}
Match → exact: True  intent: True

--- q002
GOLD: {"intent": "AGG", "frame": {"metric": "revenue", "period": "2023-Q3", "region": "North America"}}
PRED: {"intent": "AGG", "frame": {"metric": "revenue", "period": "2023-Q3", "region": "North America"}}
Match → exact: True  intent: True

--- q1001
GOLD: {"intent": "INTERVAL", "frame": {"metric": "user_retention", "start": "2018", "end": "2021"}}
PRED: {"intent": "INTERVAL", "frame": {"metric": "user_retention", "start": "2018", "end": "2021"}}
Match → ex

## 9 · Export Results & Zip for Download

Zips the LoRA adapter and eval results into a single file you can download from Drive or via `files.download()`.

In [14]:
import subprocess, shlex

# Save eval results
eval_path = ADAPTER_DIR / 'eval_results.jsonl'
with open(eval_path, 'w', encoding='utf-8') as f:
    for r in results: f.write(json.dumps(r) + '\n')
print('Eval results ->', eval_path)

# Zip adapter
zip_path = BASE_DIR / 'qwen_parser_lora_adapter.zip'
cmd = (
    f'cd {shlex.quote(str(BASE_DIR))} && '
    f'zip -r {shlex.quote(str(zip_path))} experiments/m2_e3_parse/artifacts/qwen_parser_lora'
)
subprocess.run(cmd, shell=True, check=True)
print('Zip size (MB):', round(zip_path.stat().st_size / 1e6, 1))

# Colab download
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print('Zip saved to:', zip_path)

Eval results -> /content/drive/MyDrive/snet_related/Natural-Language-Explainability-for-Temporal-KGs/experiments/m2_e3_parse/artifacts/qwen_parser_lora/eval_results.jsonl
Zip size (MB): 433.8


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Appendix A: Loading the Adapter Locally

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

ADAPTER_PATH = 'experiments/m2_e3_parse/artifacts/qwen_parser_lora'
base  = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
tok   = AutoTokenizer.from_pretrained(ADAPTER_PATH)
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()
```

## Appendix B: Merge Adapter for Faster Inference

```python
merged = model.merge_and_unload()
merged.save_pretrained('experiments/m2_e3_parse/artifacts/qwen_parser_merged')
tok.save_pretrained('experiments/m2_e3_parse/artifacts/qwen_parser_merged')
```

## Appendix C: Re-running Data Prep Locally

```bash
# Full re-run (overwrites splits_qwen/)
python scripts/m2_e3/prepare_qwen_data.py

# Dry-run — validate + print stats without writing files
python scripts/m2_e3/prepare_qwen_data.py --dry-run

# Custom ratios
python scripts/m2_e3/prepare_qwen_data.py --train-ratio 0.85 --val-ratio 0.075
```